# 02. Teacher On Held-Out BFCL Eval

This notebook evaluates the teacher model on the same held-out 50 BFCL eval examples. It does not create training data.

The goal is to estimate how strong the teacher is under the exact same harness and scoring rules that the student uses.

The output file is resume-safe. After each completed task, the result is written to disk. If the notebook stops, rerunning the eval cell skips cached task ids and continues from the missing ones.


In [1]:
from pathlib import Path
from typing import Any
import json
import sys

cwd = Path.cwd()
if (cwd / "utils.py").exists():
    blog_dir = cwd
elif (cwd / "1-distilling-a-0-8b-tool-calling-agent" / "utils.py").exists():
    blog_dir = cwd / "1-distilling-a-0-8b-tool-calling-agent"
else:
    raise RuntimeError("Run this notebook from the repo root or the blog folder.")

if str(blog_dir) not in sys.path:
    sys.path.insert(0, str(blog_dir))

import utils as u

print("Blog dir:", blog_dir)
print("Output dir:", u.OUTPUT_DIR)
print("Student model:", u.STUDENT_MODEL)
print("Teacher model:", u.TEACHER_MODEL)


Blog dir: /Users/kargarisaac/codes/personal/distillation-blogs/1-distilling-a-0-8b-tool-calling-agent
Output dir: /Users/kargarisaac/codes/personal/distillation-blogs/outputs
Student model: Qwen/Qwen3.5-0.8B
Teacher model: mlx-community/Qwen3.5-35B-A3B-4bit


In [2]:
split = u.load_bfcl_split()

print("Full BFCL v4 multi-turn base size:", len(split.all_entries))
print("Train split:", len(split.train_entries))
print("Held-out eval split:", len(split.eval_entries))
print("Current eval run:", len(split.benchmark_entries))
print("First train id:", split.train_entries[0]["id"])
print("First eval id:", split.benchmark_entries[0]["id"])
print("MAX_STEPS_PER_TURN:", u.MAX_STEPS_PER_TURN)
print("MAX_CONSECUTIVE_EXECUTION_ERRORS:", u.MAX_CONSECUTIVE_EXECUTION_ERRORS)


Full BFCL v4 multi-turn base size: 200
Train split: 150
Held-out eval split: 50
Current eval run: 50
First train id: multi_turn_base_0
First eval id: multi_turn_base_150
MAX_STEPS_PER_TURN: 20
MAX_CONSECUTIVE_EXECUTION_ERRORS: None


## Configure Teacher Endpoint

Start the raw MLX teacher server in another terminal before running the eval cell:

```bash
uv run python scripts/serve_teacher_mlx_raw.py
```

The notebook renders the full Qwen prompt locally, including the available tool schemas, sends that prompt to the raw MLX server, and receives plain generated text. Our shared Qwen XML parser then extracts tool calls. This avoids relying on the OpenAI-compatible MLX server to detect `tool_calls` for us.


In [3]:
tokenizer = u.load_tokenizer()

# Held-out pass@1 should be deterministic. Adaptive sampling stays in the optional diagnostic
# and the teacher-trace collection notebook.
teacher_config = u.TeacherConfig(temperature=0.0, top_p=1.0, top_k=0)
LOG_TRACES_TO_MLFLOW = True
mlflow_config = u.MlflowConfig(enabled=LOG_TRACES_TO_MLFLOW)
server_health = u.teacher_server_health(teacher_config)
active_teacher_model = (server_health or {}).get("model", teacher_config.model_name)

print("Teacher provider:", teacher_config.provider)
print("Teacher server:", teacher_config.server_base_url)
print("Teacher request model:", teacher_config.request_model)
print("Active teacher model:", active_teacher_model)
print("Teacher temperature:", teacher_config.temperature)
print("Teacher top_p:", teacher_config.top_p)
print("Teacher top_k:", teacher_config.top_k)
print("Teacher runtime ready:", u.teacher_runtime_is_configured(teacher_config))
print("MLflow logging enabled:", mlflow_config.enabled)
print("MLflow tracking URI:", mlflow_config.tracking_uri)
print("MLflow experiment:", mlflow_config.experiment_name)
print()
print("Cached local model candidates:")
for model_id in u.list_cached_huggingface_models():
    print(" -", model_id)
print()
print("To switch teacher model, stop the server and restart it, for example:")
print("uv run python scripts/serve_teacher_mlx_raw.py --model mlx-community/Qwen3.5-9B-MLX-8bit --port 8080")


Teacher provider: mlx_raw_server
Teacher server: http://127.0.0.1:8080
Teacher request model: default_model
Active teacher model: mlx-community/Qwen3.5-35B-A3B-4bit
Teacher temperature: 0.0
Teacher top_p: 1.0
Teacher top_k: 0
Teacher runtime ready: True
MLflow logging enabled: True
MLflow tracking URI: http://127.0.0.1:5050
MLflow experiment: distillation-blogs-bfcl

Cached local model candidates:
 - Qwen/Qwen3-4B-Instruct-2507
 - Qwen/Qwen3.5-0.8B
 - Qwen/Qwen3.5-2B
 - Qwen/Qwen3.5-35B-A3B
 - mlx-community/GLM-4.7-Flash-4bit
 - mlx-community/Qwen3.5-2B-MLX-4bit
 - mlx-community/Qwen3.5-2B-MLX-8bit
 - mlx-community/Qwen3.5-35B-A3B-4bit
 - mlx-community/Qwen3.5-4B-8bit
 - mlx-community/Qwen3.5-4B-MLX-4bit
 - mlx-community/Qwen3.5-4B-MLX-8bit
 - mlx-community/Qwen3.5-9B-4bit
 - mlx-community/Qwen3.5-9B-MLX-4bit
 - mlx-community/Qwen3.5-9B-MLX-8bit

To switch teacher model, stop the server and restart it, for example:
uv run python scripts/serve_teacher_mlx_raw.py --model mlx-community/Qw

## What We Actually Send To The Teacher Endpoint

For the teacher path, the harness first builds the exact Qwen chat-template prompt. That prompt already contains the system instruction, user message, tool schemas, previous assistant actions, and tool observations.

The raw MLX server receives only that rendered prompt plus sampling settings. It does not receive a separate `tools` field and does not try to convert model output into OpenAI-style `tool_calls`. The response is plain text, and our parser handles the Qwen XML tool-call blocks.

```mermaid
flowchart TD
  A["BFCL task state"] --> B["Harness messages"]
  C["Available tool schemas"] --> D["tokenizer.apply_chat_template"]
  B --> D
  D --> E["Rendered Qwen prompt string"]
  E --> F["POST /generate to raw MLX server"]
  F --> G["mlx_lm.stream_generate"]
  G --> H["JSON response: text, finish_reason, usage"]
  H --> I["qwen_text_from_mlx_raw_completion"]
  I --> J["Raw Qwen assistant text"]
  J --> K["parse_qwen_tool_calls"]
  K --> L{"Tool call found?"}
  L -- "yes" --> M["Convert to BFCL execution call"]
  M --> N["execute_multi_turn_func_call"]
  N --> O["Tool observation messages"]
  O --> B
  L -- "no" --> P["Stop this user turn"]
  P --> Q["score_bfcl_trace"]
```

So the call boundary is simple: the notebook calls `/generate`, gets plain text back, parses that text for `<tool_call>` blocks, executes those calls in the BFCL simulator, appends tool observations, and repeats until the model stops calling tools.


In [4]:
teacher_probe_entry = split.train_entries[0]
teacher_probe_answer = split.train_answers[0]
teacher_probe_tools = u.load_package_tools_for_example(teacher_probe_entry)
teacher_probe_chat_tools = [u.tool_for_chat_template(tool) for tool in teacher_probe_tools]
teacher_probe_schemas = u.build_tool_schema_map(teacher_probe_tools)
teacher_probe_messages = u.teacher_action_policy_messages() + [dict(message) for message in teacher_probe_entry["question"][0]]
teacher_probe_prompt = tokenizer.apply_chat_template(
    teacher_probe_messages,
    tools=teacher_probe_chat_tools,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
teacher_debug_payload = u.build_teacher_raw_completion_payload(
    prompt=teacher_probe_prompt,
    config=teacher_config,
    max_tokens=2048,
)

print("Teacher probe task id:", teacher_probe_entry["id"])
print("Endpoint URL:", f"{teacher_config.server_base_url}/generate")
print("Messages before rendering:")
print(json.dumps(teacher_probe_messages, indent=2))
print()
print("Tool count embedded in rendered prompt:", len(teacher_probe_chat_tools))
print("First tool embedded, truncated for display:")
print(json.dumps(teacher_probe_chat_tools[0], indent=2))
print()
print("Rendered Qwen prompt tail, truncated for display:")
print(teacher_debug_payload["prompt"])


Teacher probe task id: multi_turn_base_0
Endpoint URL: http://127.0.0.1:8080/generate
Messages before rendering:
[
  {
    "role": "system",
    "content": "You are an action policy agent inside a tool-use benchmark harness.\nYour job is to emit the next executable action, not to chat with the user.\nIf an available tool can advance the task, reply only with the needed <tool_call> block or blocks.\nDo not include natural-language explanation before or after tool calls.\nUse the tool schema and current conversation state to choose arguments.\nOnly answer in natural language when the task is complete or when no available tool can help."
  },
  {
    "role": "user",
    "content": "Move 'final_report.pdf' within document directory to 'temp' directory in document. Make sure to create the directory"
  }
]

Tool count embedded in rendered prompt: 32
First tool embedded, truncated for display:
{
  "name": "authenticate_twitter",
  "description": "This tool belongs to the TwitterAPI, which pro

## Teacher Dry Run On Task 1

This is the teacher-side teaching path. It prints the endpoint response, converts it into Qwen XML, parses it, executes it in BFCL, appends the simulator observation, and sends the next request. It stops when the teacher gives a final answer instead of another tool call.


In [5]:
from uuid import uuid4


def run_visible_teacher_step(
    step_index: int,
    messages: list[dict[str, str]],
    tools: list[dict[str, Any]],
    schemas: dict[str, dict[str, Any]],
    run_id: str,
) -> tuple[list[dict[str, str]], bool]:
    prompt = tokenizer.apply_chat_template(
        messages,
        tools=tools,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    payload = u.build_teacher_raw_completion_payload(prompt=prompt, config=teacher_config, max_tokens=256)

    print(f"\n=== Teacher raw endpoint round {step_index} ===")
    print("Request messages, last few before rendering:")
    print(json.dumps(messages[-6:], indent=2))
    print('-'*100)
    print("Tool count embedded in prompt:", len(tools))
    print('-'*100)
    print("Rendered prompt tail:")
    print(prompt[-2000:])
    print('-'*100)
    response_payload = u.request_mlx_raw_completion(payload, config=teacher_config)
    print("Raw endpoint response metadata:")
    print(json.dumps({key: value for key, value in response_payload.items() if key != "text"}, indent=2))

    qwen_text = u.qwen_text_from_mlx_raw_completion(response_payload)
    print('-'*100)
    print("Raw Qwen text seen by our parser:")
    print(qwen_text)
    print('-'*100)
    parse_result = u.parse_qwen_tool_calls(qwen_text)
    print("Parse errors:", parse_result.errors)
    print("Parsed calls:", parse_result.calls)
    print('-'*100)
    if not parse_result.calls:
        print("No tool call to execute. The loop stops here.")
        messages.append({"role": "assistant", "content": u.strip_generated_special_tokens(qwen_text)})
        return messages, False

    execution_calls = [u.qwen_call_to_bfcl_execution_string_for_tools(call, schemas) for call in parse_result.calls]
    print("BFCL execution calls:")
    print(execution_calls)
    print('-'*100)
    execution_results, _ = u.execute_multi_turn_func_call(
        func_call_list=execution_calls,
        initial_config=teacher_probe_entry["initial_config"],
        involved_classes=teacher_probe_entry["involved_classes"],
        model_name=run_id,
        test_entry_id=teacher_probe_entry["id"],
        long_context=False,
        is_evaL_run=False,
    )
    print('-'*100)
    print("Simulator observations appended as tool messages:")
    print(json.dumps(execution_results, indent=2))

    messages.append({"role": "assistant", "content": u.strip_generated_special_tokens(qwen_text)})
    messages.extend(u.render_tool_response_messages(execution_results))
    return messages, True


if not u.teacher_runtime_is_configured(teacher_config):
    print("Teacher server is not ready. Start it with: uv run python scripts/serve_teacher_mlx_raw.py")
else:
    teacher_visible_messages = [dict(message) for message in teacher_probe_messages]
    teacher_visible_run_id = f"teacher_visible_{teacher_probe_entry['id']}_{uuid4().hex[:8]}"
    VISIBLE_TEACHER_DEBUG_STEPS = 10

    for visible_step_index in range(1, VISIBLE_TEACHER_DEBUG_STEPS + 1):
        teacher_visible_messages, should_continue = run_visible_teacher_step(
            visible_step_index,
            teacher_visible_messages,
            teacher_probe_chat_tools,
            teacher_probe_schemas,
            teacher_visible_run_id,
        )
        if not should_continue:
            break



=== Teacher raw endpoint round 1 ===
Request messages, last few before rendering:
[
  {
    "role": "system",
    "content": "You are an action policy agent inside a tool-use benchmark harness.\nYour job is to emit the next executable action, not to chat with the user.\nIf an available tool can advance the task, reply only with the needed <tool_call> block or blocks.\nDo not include natural-language explanation before or after tool calls.\nUse the tool schema and current conversation state to choose arguments.\nOnly answer in natural language when the task is complete or when no available tool can help."
  },
  {
    "role": "user",
    "content": "Move 'final_report.pdf' within document directory to 'temp' directory in document. Make sure to create the directory"
  }
]
----------------------------------------------------------------------------------------------------
Tool count embedded in prompt: 32
-----------------------------------------------------------------------------------

## Run Teacher Eval On 50 Held-Out Examples

This uses the held-out eval split. It is for reporting teacher strength, not for creating distillation rows.


In [ ]:
if not u.teacher_runtime_is_configured(teacher_config):
    print("Teacher server is not ready. Start it with: uv run python scripts/serve_teacher_mlx_raw.py")
else:
    generate_teacher_action = u.make_teacher_action_generator(teacher_config)
    teacher_slug = u.model_slug(active_teacher_model)
    teacher_eval_path = u.OUTPUT_DIR / f"{teacher_slug}_bfcl_v4_multi_turn_base_eval_50_raw_qwen_parser_v2_full_trace_pass1.json"
    teacher_eval = u.evaluate_entries(
        split.benchmark_entries,
        split.benchmark_answers,
        tokenizer=tokenizer,
        run_name_prefix="qwen_35b_teacher_eval_pass1",
        generate_action=generate_teacher_action,
        output_path=teacher_eval_path,
        capture_prompts=True,
        prompt_messages_prefix=u.teacher_action_policy_messages(),
        mlflow_config=mlflow_config,
        mlflow_tags={"model_role": "teacher", "model_name": active_teacher_model, "provider": teacher_config.provider},
    )

    print("\nTeacher pass@1 accuracy:", teacher_eval["accuracy"])
    print(f"Correct: {teacher_eval['correct_count']}/{teacher_eval['total_count']}")
    print("Saved to:", teacher_eval_path)


Loaded 16 cached eval results from: /Users/kargarisaac/codes/personal/distillation-blogs/outputs/mlx_community_Qwen3_5_35B_A3B_4bit_bfcl_v4_multi_turn_base_eval_50_raw_qwen_parser_v2_full_trace_pass1.json
Skipping multi_turn_base_150 (cached)
Skipping multi_turn_base_151 (cached)
Skipping multi_turn_base_152 (cached)
Skipping multi_turn_base_153 (cached)
Skipping multi_turn_base_154 (cached)
Skipping multi_turn_base_155 (cached)
Skipping multi_turn_base_156 (cached)
Skipping multi_turn_base_157 (cached)
Skipping multi_turn_base_158 (cached)
Skipping multi_turn_base_159 (cached)
Skipping multi_turn_base_160 (cached)
Skipping multi_turn_base_161 (cached)
Skipping multi_turn_base_162 (cached)
Skipping multi_turn_base_163 (cached)
Skipping multi_turn_base_164 (cached)
Skipping multi_turn_base_165 (cached)
Running multi_turn_base_166...
  score: valid
🏃 View run qwen_35b_teacher_eval_pass1_dc226452_multi_turn_base_166 at: http://127.0.0.1:5050/#/experiments/5/runs/767ffdb7a528490f82a23b5dbf

## Optional Teacher Adaptive Pass@5 Diagnostic

Teacher pass@1 is the main held-out eval number. Adaptive pass@5 is optional because it evaluates the teacher inside a search policy. That can be useful for understanding how much the scorer can rescue sampled trajectories, but it should not replace pass@1.


In [ ]:
RUN_TEACHER_ADAPTIVE_EVAL = False

if not RUN_TEACHER_ADAPTIVE_EVAL:
    print("Skipped optional teacher adaptive pass@5 eval. Set RUN_TEACHER_ADAPTIVE_EVAL = True to run it.")
elif not u.teacher_runtime_is_configured(teacher_config):
    print("Teacher server is not ready. Start it with: uv run python scripts/serve_teacher_mlx_raw.py")
else:
    generate_teacher_action_factory = u.make_teacher_action_generator_factory(teacher_config)
    teacher_slug = u.model_slug(active_teacher_model)
    teacher_adaptive_eval_path = u.OUTPUT_DIR / f"{teacher_slug}_bfcl_v4_multi_turn_base_eval_50_raw_qwen_parser_v2_full_trace_adaptive_pass5.json"
    teacher_attempt_preview = u.adaptive_temperature_retry_attempts_for_task(0)
    print("Teacher adaptive eval retry attempts per task:", len(teacher_attempt_preview))
    print("Retry schedule:")
    print(json.dumps([u.attempt_config_to_dict(attempt) for attempt in teacher_attempt_preview], indent=2))

    teacher_adaptive_eval = u.evaluate_entries(
        split.benchmark_entries,
        split.benchmark_answers,
        tokenizer=tokenizer,
        run_name_prefix="qwen_35b_teacher_eval_adaptive_pass5",
        generate_action_factory=generate_teacher_action_factory,
        attempt_configs_factory=u.adaptive_temperature_retry_attempts_for_task,
        output_path=teacher_adaptive_eval_path,
        capture_prompts=True,
        mlflow_config=mlflow_config,
        mlflow_tags={"model_role": "teacher", "model_name": active_teacher_model, "provider": teacher_config.provider, "eval_mode": "adaptive_pass5"},
        prompt_messages_prefix=u.teacher_action_policy_messages(),
    )

    print("\nTeacher adaptive pass@5 accuracy:", teacher_adaptive_eval["accuracy"])
    print(f"Correct: {teacher_adaptive_eval['correct_count']}/{teacher_adaptive_eval['total_count']}")
    print(f"First-attempt correct: {teacher_adaptive_eval['first_attempt_correct_count']}/{teacher_adaptive_eval['total_count']}")
    print("Saved to:", teacher_adaptive_eval_path)
